# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriKeerthi16-b/Machine-Learning/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# Clone repository if it is not already available
if not os.path.exists("Machine-Learning"):
    !git clone https://github.com/SriKeerthi16-b/Machine-Learning.git

# Load dataset
DATA_PATH = "Machine-Learning/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# Create target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Final feature set
feature_cols = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Keep usable rows
model_df = df.dropna(
    subset=feature_cols + ["is_declining_label", "client_id"]
).copy()

X = model_df[feature_cols]
y = model_df["is_declining_label"]
groups = model_df["client_id"]

# Client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train model
model = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

# Generate model scores
test_scores = model.predict_proba(X_test)[:, 1]

print("Dataset rows:", len(df))
print("Usable modeling rows:", len(model_df))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Client overlap:",
      len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

Cloning into 'Machine-Learning'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 171 (delta 73), reused 98 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (171/171), 1.89 MiB | 13.46 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Dataset rows: 30000
Usable modeling rows: 22301
Training rows: 17223
Test rows: 5078
Client overlap: 0


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create a human-readable ranked action queue

queue = model_df.loc[X_test.index].copy()

queue["model_score"] = test_scores

def make_reason(row):
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("STALE_CONTENT")

    if row["impressions_90d"] >= 500:
        reasons.append("HIGH_VISIBILITY")

    if row["avg_position"] >= 10:
        reasons.append("LOWER_RANKING")

    if row["ctr"] < 0.03:
        reasons.append("LOW_CTR")

    if row["word_count"] < 500:
        reasons.append("SHORT_CONTENT")

    if not reasons:
        reasons.append("MODEL_SIGNAL")

    return ", ".join(reasons[:3])


queue["reason_code"] = queue.apply(make_reason, axis=1)

queue["priority"] = pd.qcut(
    queue["model_score"].rank(method="first"),
    4,
    labels=["Low", "Medium", "High", "Critical"]
)

queue = queue.sort_values(
    "model_score",
    ascending=False
)

display(
    queue[
        [
            "model_score",
            "priority",
            "reason_code",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "word_count"
        ]
    ].head(20)
)

,model_score,priority,reason_code,days_since_last_update,impressions_90d,avg_position,ctr,word_count
13,0.683416,High,"LOWER_RANKING, LOW_CTR",103,307,39.8,0.00,1342.0
29990,0.683416,Critical,"HIGH_VISIBILITY, LOWER_RANKING",14,2542,22.5,0.16,2501.0
22,0.683416,High,HIGH_VISIBILITY,7,2311,3.9,0.13,3058.0
26,0.683416,High,"HIGH_VISIBILITY, LOWER_RANKING",13,2426,30.0,0.12,2686.0
29866,0.683416,Critical,"HIGH_VISIBILITY, LOWER_RANKING",106,2243,24.2,0.04,2837.0
43,0.683416,High,LOW_CTR,20,184,2.9,0.00,2394.0
29850,0.683416,Critical,LOW_CTR,20,151,6.3,0.00,2865.0
29849,0.683416,Critical,"HIGH_VISIBILITY, LOWER_RANKING",25,4449,22.3,0.16,2495.0
11140,0.683416,Critical,"HIGH_VISIBILITY, LOWER_RANKING",20,4872,19.3,0.12,2625.0
11141,0.683416,Critical,HIGH_VISIBILITY,20,11442,4.8,0.24,2689.0


The queue ranks pages by the model's estimated likelihood of decline, with the highest-priority pages shown first. Each recommendation includes a short reason code based on observable signals so that a human reviewer can understand why a page was prioritized. The queue is intended for review and prioritization, not automatic content changes.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic summary of the recommendation queue

print("Total pages in queue:", len(queue))
print("Critical:", (queue["priority"] == "Critical").sum())
print("High:", (queue["priority"] == "High").sum())
print("Medium:", (queue["priority"] == "Medium").sum())
print("Low:", (queue["priority"] == "Low").sum())

print("\nHighest model score:",
      round(queue["model_score"].max(), 3))

print("Lowest model score:",
      round(queue["model_score"].min(), 3))

Total pages in queue: 5078
Critical: 1270
High: 1269
Medium: 1269
Low: 1270

Highest model score: 0.683
Lowest model score: 0.003


This queue is intended for SEO/content teams to prioritize pages for human review. A reviewer can use the ranking and reason codes to decide which pages deserve attention first. The output is decision-support rather than an automatic publishing or editing system.

The approach is limited to the dataset, features, validation design, and observed patterns used here. It should not be treated as a guarantee of future traffic changes. It may become less reliable when the underlying search environment, content mix, client population, or data distribution changes substantially.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Human-review checklist

review_checklist = [
    "Check current page content and search intent",
    "Check whether the page is still relevant",
    "Check recent content or ranking changes",
    "Review the signals behind the reason code",
    "Confirm that an action is appropriate before making changes"
]

no_go_actions = [
    "Automatic publishing",
    "Automatic deletion",
    "Automatic redirects",
    "Automatic major content rewrites",
    "Automatic claims about future traffic or penalties"
]

print("HUMAN REVIEW CHECKLIST")
for item in review_checklist:
    print("-", item)

print("\nNO-GO AUTOMATION LIST")
for item in no_go_actions:
    print("-", item)

HUMAN REVIEW CHECKLIST
- Check current page content and search intent
- Check whether the page is still relevant
- Check recent content or ranking changes
- Review the signals behind the reason code
- Confirm that an action is appropriate before making changes

NO-GO AUTOMATION LIST
- Automatic publishing
- Automatic deletion
- Automatic redirects
- Automatic major content rewrites
- Automatic claims about future traffic or penalties


Before acting on a recommendation, a human reviewer should check the page's current content, search intent, relevance, recent updates, ranking context, and whether there is a legitimate reason for the observed signals. The reason code is only an explanation of observed signals and should not be treated as proof that a page needs a specific change.

The system should never automatically publish, delete, rewrite, redirect, or substantially change content based only on the model score. It should also not make claims about business impact, search-engine penalties, or future traffic without human and domain review.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Current monitoring snapshot

monitoring = pd.DataFrame({
    "feature": feature_cols,
    "missing_rate": [
        model_df[col].isna().mean()
        for col in feature_cols
    ],
    "mean": [
        model_df[col].mean()
        for col in feature_cols
    ]
})

display(monitoring)

print(
    "\nProposed trigger: investigate retraining if measured "
    "Precision@50 falls materially below the validated result."
)

print(
    "Proposed trigger: investigate data drift if feature "
    "distributions or missingness change substantially."
)

,feature,missing_rate,mean
0,content_age_days,0.0,209.080041
1,days_since_last_update,0.0,41.893637
2,impressions_90d,0.0,5078.628985
3,avg_position,0.0,15.358226
4,ctr,0.0,0.598717
5,word_count,0.0,3107.760325



Proposed trigger: investigate retraining if measured Precision@50 falls materially below the validated result.
Proposed trigger: investigate data drift if feature distributions or missingness change substantially.


The recommendations should be reconsidered if the input data or measured performance changes materially. Proposed triggers are a sustained drop in Precision@50, a noticeable change in the distribution of important features, a large increase in missing values, or evidence that the model is being applied to data unlike the training data. These are monitoring triggers for review, not claims that drift has already occurred.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

export_cols = [
    "model_score",
    "priority",
    "reason_code",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

queue_export = queue[export_cols].copy()

output_path = os.path.join(
    output_dir,
    "ranked_action_queue.csv"
)

queue_export.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows exported:", len(queue_export))

display(queue_export.head(10))

Saved: work/outputs/ranked_action_queue.csv
Rows exported: 5078


,model_score,priority,reason_code,days_since_last_update,impressions_90d,avg_position,ctr,word_count
13,0.683416,High,"LOWER_RANKING, LOW_CTR",103,307,39.8,0.00,1342.0
29990,0.683416,Critical,"HIGH_VISIBILITY, LOWER_RANKING",14,2542,22.5,0.16,2501.0
22,0.683416,High,HIGH_VISIBILITY,7,2311,3.9,0.13,3058.0
26,0.683416,High,"HIGH_VISIBILITY, LOWER_RANKING",13,2426,30.0,0.12,2686.0
29866,0.683416,Critical,"HIGH_VISIBILITY, LOWER_RANKING",106,2243,24.2,0.04,2837.0
43,0.683416,High,LOW_CTR,20,184,2.9,0.00,2394.0
29850,0.683416,Critical,LOW_CTR,20,151,6.3,0.00,2865.0
29849,0.683416,Critical,"HIGH_VISIBILITY, LOWER_RANKING",25,4449,22.3,0.16,2495.0
11140,0.683416,Critical,"HIGH_VISIBILITY, LOWER_RANKING",20,4872,19.3,0.12,2625.0
11141,0.683416,Critical,HIGH_VISIBILITY,20,11442,4.8,0.24,2689.0


I export the ranked decision-support queue so that the paper can reuse the measured results without rerunning the ranking manually. The exported file contains model scores, priority levels, reason codes, and the observable signals used for review. It does not contain client names, private queries, or other private identifiers.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.